In [54]:
# tester for feature builder cleaning

In [55]:
# =========================
# 0) Imports, engine check, folders
# =========================

# Core data stack + utilities; warnings silenced for cleaner notebook logs
import pandas as pd, numpy as np, warnings, re
from pathlib import Path
warnings.filterwarnings('ignore')

import pyarrow  
PARQUET_ENGINE = "pyarrow"


# Project I/O roots:
# - RAW: expected location for source files (CSV)
# - PROC: destination for cleaned/engineered outputs and intermediates
RAW  = Path('./raw')            
PROC = Path('./processed'); PROC.mkdir(exist_ok=True)  

CPI processing

# =========================
# CPI: Monthly -> YoY -> Quarterly (final)
# =========================

# Read CPI data (monthly frequency expected)


# --- helpers (minimal but robust) ---

def _pick(df: pd.DataFrame, candidates) -> str | None:
    """Return the first matching column from `candidates` in df (exact, else substring)."""
    for c in candidates:
        if c in df.columns:
            return c
    for col in df.columns:
        for k in candidates:
            if k.lower() in col.lower():
                return col
    return None

def _smart_parse_dates(s: pd.Series) -> pd.Series:
    """
    Try several parse strategies and pick the one that yields
    the most complete monthly coverage.
    """
    s = s.astype(str)
    attempts = [
        ('infer',    pd.to_datetime(s, errors='coerce')),
        ('dayfirst', pd.to_datetime(s, errors='coerce', dayfirst=True)),
    ]
    fmts = ['%d-%b-%Y','%d-%b-%y','%d/%m/%Y','%m/%d/%Y','%Y-%m-%d',
            '%b %d, %Y','%d %b %Y','%b %Y','%m-%Y']
    attempts += [(fmt, pd.to_datetime(s, format=fmt, errors='coerce')) for fmt in fmts]

    def _score(dt: pd.Series):
        ok = dt.dropna()
        if ok.empty:
            return (0, 0, 0)
        ser = pd.Series(1, index=ok).sort_index()
        by_year = ser.groupby(ser.index.year).apply(lambda x: len(pd.Index(x.index.month).unique()))
        med_months = int(by_year.median()) if len(by_year) else 0
        return (len(ok), med_months, len(pd.Index(ok.dt.month).unique()))

    best_name, best_dt, _ = max(((n, d, _score(d)) for n, d in attempts), key=lambda t: t[2])
    # Optional: print(f"[CPI] date parse picked: {best_name}")
    return best_dt

def _parse_dates_in_df(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    out = df.copy()
    out[date_col] = _smart_parse_dates(out[date_col])
    return out.dropna(subset=[date_col]).sort_values(date_col)

def _yoy_from_monthly(series: pd.Series) -> pd.Series:
    """YoY % from a monthly level series (requires 12 months)."""
    return series.pct_change(12) * 100.0

def _monthly_to_quarter(series: pd.Series, how: str = 'mean') -> pd.Series:
    """Aggregate monthly series to quarter-end with mean (default) or last."""
    return series.resample('Q').mean() if how == 'mean' else series.resample('Q').last()


# --- main builder ---

def build_cpi_quarterly_yoy(
    csv_path: str,
    date_candidates = ('DATE','Date','Month','Period'),
    value_candidates = ('CPI','Index','Value','CPI_COMBINED_RAW2012_100'),
    quarterly_agg: str = 'mean'
) -> tuple[pd.Series, pd.Series, pd.Series]:
    """
    Returns:
      cpi_index_m : monthly CPI index (level)
      cpi_yoy_m   : monthly CPI YoY (%)
      cpi_yoy_q   : quarterly CPI YoY (%)  -> column name 'cpi_yoy'
    """
    raw = pd.read_csv(csv_path)
    dcol = _pick(raw, date_candidates)
    vcol = _pick(raw, value_candidates)
    if not dcol or not vcol:
        raise KeyError(f"CPI columns not found. Have={list(raw.columns)} | "
                       f"need date∈{date_candidates}, value∈{value_candidates}")

    # parse and index by date
    tidy = _parse_dates_in_df(raw, dcol)
    cpi_index_m = tidy.set_index(dcol)[vcol].astype(float)

    # ensure monthly frequency (month-end)
    cpi_index_m = cpi_index_m.asfreq('M', method='pad')  # or .resample('M').last()

    # YoY (%), then quarterly mean
    cpi_yoy_m = _yoy_from_monthly(cpi_index_m).rename('cpi_yoy_m')
    cpi_yoy_q = _monthly_to_quarter(cpi_yoy_m, how=quarterly_agg).rename('cpi_yoy')

    # diagnostics
    # print("CPI monthly:", cpi_index_m.index.min().date(), "→", cpi_index_m.index.max().date())
    # print("CPI YoY (M) non-null:", cpi_yoy_m.dropna().shape[0])
    # print("CPI YoY (Q) non-null:", cpi_yoy_q.dropna().shape[0])

    return cpi_index_m, cpi_yoy_m, cpi_yoy_q


# --- usage ---
cpi_index_m, cpi_yoy_m, cpi_yoy_q = build_cpi_quarterly_yoy(RAW/'CPI_Monthly_Jan_2013_to_Jun_2025.csv')  
# # For your modeling join, use: cpi_yoy_q (name == 'cpi_yoy')

cpi_index_m,
cpi_yoy_m,
cpi_yoy_q

GDP


# --- GDP quarterly loader (fixes '2012.0-Q1' etc.) ---

import re
from pandas.tseries.offsets import QuarterEnd  # imported if you later need explicit quarter-end arithmetic

# Read the raw GDP table
gdp_raw = pd.read_csv(RAW/'GDP_Quarterly_2010_2025.csv').copy()

# 1) Clean numeric column
# Expect a GDP "level" column named 'gdp'; coerce strings like "1,234.5" into float.
if 'gdp' not in gdp_raw.columns:
    raise ValueError("Expected a 'gdp' column in GDP_Quarterly_2010_2025.csv")

gdp_raw['gdp_clean'] = pd.to_numeric(
    gdp_raw['gdp'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)

# 2) Normalize quarter strings (strip the '.0' before '-Q')
# Expect some variant of a quarter label in a column named 'quarter'.
if 'quarter' not in gdp_raw.columns:
    raise ValueError("Expected a 'quarter' column in GDP_Quarterly_2010_2025.csv")

# Remove trailing ".0" artifacts that sometimes appear after numeric-to-string conversions.
gdp_raw['quarter_str'] = gdp_raw['quarter'].astype(str).str.replace('.0', '', regex=False)

# 3) Extract YYYY and Qn from anything like '2012-Q1', '2012Q1', 'Q1 2012.0', etc.
# Use a permissive regex to find 4-digit year followed by a 'Q[1-4]' anywhere in the string.
m = gdp_raw['quarter_str'].str.extract(r'(?P<y>\d{4}).*?Q(?P<q>[1-4])')

# Build canonical tokens like "2012Q1" and mark valid rows
qstr = (m['y'].fillna('') + 'Q' + m['q'].fillna(''))
valid = qstr.str.match(r'^\d{4}Q[1-4]$')

# Keep only rows that parse into a clean Year+Quarter token
gdp_norm = gdp_raw.loc[valid].copy()

# Convert to quarter-end timestamps using PeriodIndex (calendar quarters)
gdp_norm['__q'] = pd.PeriodIndex(qstr[valid], freq='Q').to_timestamp(how='end')

# 4) Build a clean quarterly level series (last obs per quarter if duplicates)
# If multiple rows map to the same quarter, take the last one (common when data is revised).
gdp_q_level = (gdp_norm
               .dropna(subset=['__q','gdp_clean'])
               .set_index('__q')
               .sort_index()
               .groupby(pd.Grouper(freq='Q'))['gdp_clean']
               .last())

# 5) Compute YoY from levels (4-quarter change * 100)
# This assumes levels are comparable across quarters (same base/deflator).
gdp_q = gdp_q_level.pct_change(4).mul(100.0).rename('gdp_yoy').dropna()

# 6) Diagnostics
print("Unique 'quarter' samples:", gdp_raw['quarter'].astype(str).unique()[:8])
print("Normalized samples:", gdp_norm['quarter_str'].astype(str).unique()[:8])
print("GDP quarterly points:", gdp_q.shape[0])
if not gdp_q.empty:
    print("GDP range:", gdp_q.index.min().date(), "→", gdp_q.index.max().date())
display(gdp_q.head(4))

Repo rate

# ---------- Helpers ----------

def pick(df, candidates):
    for c in candidates:
        if c in df.columns: 
            return c
    for c in df.columns:
        for k in candidates:
            if k.lower() in c.lower(): 
                return c
    return None

def smart_parse_dates(s: pd.Series) -> pd.Series:
    s = s.astype(str)
    attempts = [
        ('infer',    pd.to_datetime(s, errors='coerce')),
        ('dayfirst', pd.to_datetime(s, errors='coerce', dayfirst=True)),
    ]
    fmts = ['%d-%b-%Y','%d-%b-%y','%d/%m/%Y','%m/%d/%Y','%Y-%m-%d',
            '%b %d, %Y','%d %b %Y','%b %Y','%m-%Y']
    attempts += [(fmt, pd.to_datetime(s, format=fmt, errors='coerce')) for fmt in fmts]

    def score(dt):
        ok = dt.dropna()
        if ok.empty: 
            return (0, 0, 0)
        ser = pd.Series(1, index=ok).sort_index()
        by_year_months = ser.groupby(ser.index.year).apply(
            lambda x: len(pd.Index(x.index.month).unique())
        )
        med_months = int(by_year_months.median()) if len(by_year_months) else 0
        return (len(ok), med_months, len(pd.Index(ok.dt.month).unique()))

    best_name, best_dt, best_score = max(
        [(n, d, score(d)) for n, d in attempts], key=lambda t: t[2]
    )
    print(f"[smart_parse_dates] picked: {best_name} | parsed={best_score[0]} "
          f"| median months/yr={best_score[1]} | unique months={best_score[2]}")
    return best_dt

def parse_dates_in_df(df, date_col='Date'):
    out = df.copy()
    out[date_col] = smart_parse_dates(out[date_col])
    out = out.dropna(subset=[date_col]).sort_values(date_col)
    return out

def load_repo_monthly_to_quarter(path: Path):
    raw = pd.read_csv(path)
    print("Repo raw columns:", list(raw.columns))

    # 1) find date & rate columns
    dcol = pick(raw, ['DATE','Date','Month','Period'])
    vcol = pick(raw, ['REPO_RATE_PERCENT','Repo','Rate','Policy Rate','REPO','Value'])
    print("Detected -> date:", dcol, "| value:", vcol)
    if dcol is None or vcol is None:
        raise ValueError("Repo CSV must have a date column (DATE/Month/Period) and a value column (Repo/Rate/REPO_RATE_PERCENT).")

    # 2) parse dates & clean numeric
    df = parse_dates_in_df(raw, dcol)

    val = (df[vcol].astype(str)
                  .str.replace('\u200b', '', regex=False)
                  .str.replace(',', '', regex=False)
                  .str.replace('%', '', regex=False)
                  .str.strip())
    val = pd.to_numeric(val, errors='coerce')

    # 3) detect scaling (percent vs fraction)
    median_abs = val.dropna().abs().median()
    print("Median(raw value) =", median_abs)
    if pd.notna(median_abs) and median_abs < 1:
        val = val * 100.0
        print("Scaled by 100 (values looked like fractions).")

    df = df.assign(val=val).dropna(subset=['val'])
    if df.empty:
        raise ValueError("Repo series is empty after cleaning; check the source file/headers.")

    # 4) normalize to month-end stamps, forward-fill to complete monthly index
    m_end = (df.set_index(dcol)['val']
               .to_period('M').to_timestamp('M')
               .sort_index())
    full_m = pd.date_range(m_end.index.min(), m_end.index.max(), freq='M')
    m_end = m_end.reindex(full_m).ffill()

    # 5) quarter-end level & QoQ change in bps
    repo_q = m_end.resample('Q').last().rename('repo')                 # % at quarter-end
    repo_chg_bps = (repo_q.diff() * 100.0).rename('repo_chg_bps')      # Δ in basis points

    # 6) coverage diagnostics & previews
    print("Coverage:", m_end.index.min().date(), "→", m_end.index.max().date(),
          "| months:", len(m_end), "| quarters:", len(repo_q.dropna()))

    # Prepare previews
    head_q = repo_q.head(6).to_frame().reset_index().rename(columns={'index':'QuarterEnd'})
    tail_q = repo_q.tail(6).to_frame().reset_index().rename(columns={'index':'QuarterEnd'})
    head_d = df.head(6).reset_index(drop=True)
    tail_d = df.tail(6).reset_index(drop=True)

    print("Repo (raw cleaned) head", head_d)
    print("Repo (raw cleaned) tail", tail_d)
    print("Repo quarterly level (head)", head_q)
    print("Repo quarterly level (tail)", tail_q)

    return repo_q, repo_chg_bps

repo_q, repo_chg_bps = load_repo_monthly_to_quarter("raw/Repo_Rate_Monthly_2010_2025.csv")

print("Repo quarterly last 5:")
print(repo_q.tail())
print("Repo Δbps quarterly last 5:")
print(repo_chg_bps.tail())

Nifty 50 and NiftyMidcap100

# ---------- Helpers (from user's snippet, slightly adapted) ----------

def pick(df, candidates):
    for c in candidates:
        if c in df.columns: 
            return c
    for c in df.columns:
        for k in candidates:
            if k.lower() in c.lower(): 
                return c
    return None

def smart_parse_dates(s: pd.Series) -> pd.Series:
    s = s.astype(str)
    attempts = []
    attempts.append(('infer', pd.to_datetime(s, errors='coerce')))
    attempts.append(('dayfirst', pd.to_datetime(s, errors='coerce', dayfirst=True)))
    fmts = ['%d-%b-%Y','%d-%b-%y','%d/%m/%Y','%m/%d/%Y','%Y-%m-%d',
            '%b %d, %Y','%d %b %Y','%b %Y','%m-%Y']
    attempts += [(fmt, pd.to_datetime(s, format=fmt, errors='coerce')) for fmt in fmts]
    def score(dt):
        ok = dt.dropna()
        if ok.empty: 
            return (0, 0, 0)
        ser = pd.Series(1, index=ok).sort_index()
        by_year_months = ser.groupby(ser.index.year).apply(lambda x: len(pd.Index(x.index.month).unique()))
        med_months = int(by_year_months.median()) if len(by_year_months) else 0
        return (len(ok), med_months, len(pd.Index(ok.dt.month).unique()))
    best_name, best_dt, best_score = max([(n, d, score(d)) for n, d in attempts], key=lambda t: t[2])
    print(f"[smart_parse_dates] picked: {best_name} | parsed={best_score[0]} | median months/yr={best_score[1]} | unique months={best_score[2]}")
    return best_dt

def parse_dates_in_df(df, date_col='Date'):
    out = df.copy()
    out[date_col] = smart_parse_dates(out[date_col])
    out = out.dropna(subset=[date_col]).sort_values(date_col)
    return out

def _parse_investing_prices(df, date_col='Date', price_col='Price'):
    out = parse_dates_in_df(df, date_col=date_col)
    out[price_col] = pd.to_numeric(out[price_col].astype(str).str.replace(',', '', regex=False), errors='coerce')
    out = out.dropna(subset=[price_col])
    return out[[date_col, price_col]]

def investing_to_quarter_ret(df, date_col='Date', price_col='Price'):
    df2 = _parse_investing_prices(df, date_col, price_col)
    q_end_price = df2.set_index(date_col)[price_col].resample('Q').last()
    return q_end_price.pct_change()

def check_monthly_coverage(df, date_col='Date', label='series'):
    dates = smart_parse_dates(df[date_col]).dropna()
    ser = pd.Series(1, index=dates).sort_index()
    months_per_year = ser.groupby(ser.index.year).apply(lambda s: sorted(pd.Index(s.index.month).unique()))
    print(f"[{label}] Months present per year:")
    for y, months in months_per_year.items():
        print(f"  {y}: {months}")
    q_counts = ser.resample('Q').size()
    miss = q_counts[q_counts == 0]
    if len(miss) > 0:
        print(f"[{label}] ⚠ Missing {len(miss)} quarter(s) with zero rows — resample will yield NaNs.")
    else:
        print(f"[{label}] ✅ At least one row in every quarter.")

# ---------- Load the uploaded CSVs ----------
nifty50_path = "raw/Nifty50.csv"
midcap_path  = "raw/NIFTYMidcap100.csv"

n50_raw = pd.read_csv(nifty50_path)
mid_raw = pd.read_csv(midcap_path)

print("NIFTY50 columns:", list(n50_raw.columns))
print("MIDCAP100 columns:", list(mid_raw.columns))

# Detect date & price columns (more candidates for robustness)
date_candidates  = ['Date','DATE','date']
price_candidates = ['Price','Close','Adj Close','AdjClose','Last','PX_LAST']

n50_date = pick(n50_raw, date_candidates)
n50_price = pick(n50_raw, price_candidates)
mid_date = pick(mid_raw, date_candidates)
mid_price = pick(mid_raw, price_candidates)

print("Detected columns -> NIFTY50: date =", n50_date, ", price =", n50_price)
print("Detected columns -> MIDCAP100: date =", mid_date, ", price =", mid_price)

# Coverage QA
check_monthly_coverage(n50_raw, n50_date, 'NIFTY 50')
check_monthly_coverage(mid_raw, mid_date, 'NIFTY Midcap 100')

# Build quarterly returns using detected columns
nifty_qret  = investing_to_quarter_ret(n50_raw, date_col=n50_date, price_col=n50_price).rename('nifty_qret')
midcap_qret = investing_to_quarter_ret(mid_raw, date_col=mid_date, price_col=mid_price).rename('midcap_qret')

# Compute excess (aligned by DatetimeIndex)
excess_ret = (midcap_qret - nifty_qret).rename('excess_ret')

# Diagnostics
print("Quarters with NIFTY qret:", nifty_qret.dropna().shape[0],
      "| range:", nifty_qret.dropna().index.min().date(), "→", nifty_qret.dropna().index.max().date())
print("Quarters with MIDCAP qret:", midcap_qret.dropna().shape[0],
      "| range:", midcap_qret.dropna().index.min().date(), "→", midcap_qret.dropna().index.max().date())
print("Quarters with EXCESS:", excess_ret.dropna().shape[0],
      "| range:", excess_ret.dropna().index.min().date(), "→", excess_ret.dropna().index.max().date())

# Show previews
print("NIFTY50 quarterly returns (head)", nifty_qret.dropna().head(10).to_frame().reset_index().rename(columns={'index':'QuarterEnd'}))
print("MIDCAP100 quarterly returns (head)", midcap_qret.dropna().head(10).to_frame().reset_index().rename(columns={'index':'QuarterEnd'}))
print("Excess (Midcap - NIFTY) quarterly returns (head)", excess_ret.dropna().head(10).to_frame().reset_index().rename(columns={'index':'QuarterEnd'}))

print("NIFTY50 quarterly returns (tail)", nifty_qret.dropna().tail(10).to_frame().reset_index().rename(columns={'index':'QuarterEnd'}))
print("MIDCAP100 quarterly returns (tail)", midcap_qret.dropna().tail(10).to_frame().reset_index().rename(columns={'index':'QuarterEnd'}))
print("Excess (Midcap - NIFTY) quarterly returns (tail)", excess_ret.dropna().tail(10).to_frame().reset_index().rename(columns={'index':'QuarterEnd'}))

Merged work

In [56]:
# Build standardized, quarter-labeled CSVs for: excess_ret, cpi_yoy, gdp_yoy, repo (level), repo_chg_bps
# and produce both OUTER and INNER merged files to inspect coverage/alignment.

import pandas as pd
import numpy as np
from pathlib import Path

RAW  = Path('./raw')            
PROC = Path('./processed') 

# ---------- Helpers (lite) ----------

def _pick(df: pd.DataFrame, candidates) -> str | None:
    for c in candidates:
        if c in df.columns: 
            return c
    for col in df.columns:
        for k in candidates:
            if k.lower() in col.lower(): 
                return col
    return None

def _smart_parse_dates(s: pd.Series) -> pd.Series:
    s = s.astype(str)
    attempts = [
        ('infer',    pd.to_datetime(s, errors='coerce')),
        ('dayfirst', pd.to_datetime(s, errors='coerce', dayfirst=True)),
    ]
    fmts = ['%d-%b-%Y','%d-%b-%y','%d/%m/%Y','%m/%d/%Y','%Y-%m-%d',
            '%b %d, %Y','%d %b %Y','%b %Y','%m-%Y']
    attempts += [(fmt, pd.to_datetime(s, format=fmt, errors='coerce')) for fmt in fmts]

    def _score(dt):
        ok = dt.dropna()
        if ok.empty:
            return (0, 0, 0)
        ser = pd.Series(1, index=ok).sort_index()
        by_year = ser.groupby(ser.index.year).apply(lambda x: len(pd.Index(x.index.month).unique()))
        med_months = int(by_year.median()) if len(by_year) else 0
        return (len(ok), med_months, len(pd.Index(ok.dt.month).unique()))

    best_name, best_dt, _ = max(((n, d, _score(d)) for n, d in attempts), key=lambda t: t[2])
    return best_dt

def _parse_dates_in_df(df, date_col: str):
    out = df.copy()
    out[date_col] = _smart_parse_dates(out[date_col])
    return out.dropna(subset=[date_col]).sort_values(date_col)

def _monthly_to_quarter(series: pd.Series, how='mean'):
    return series.resample('Q').mean() if how=='mean' else series.resample('Q').last()

def _yoy_from_monthly(series: pd.Series) -> pd.Series:
    return series.pct_change(12) * 100.0

def _standardize_q_series(s: pd.Series, name: str) -> pd.DataFrame:
    """Take a quarterly series indexed by quarter-end Timestamp and return a labeled DataFrame."""
    s = s.dropna().copy()
    # Normalize to calendar quarter-end timestamps
    s.index = s.index.to_period('Q').to_timestamp('Q')
    per = s.index.to_period('Q')
    out = pd.DataFrame({
        "quarter_end": s.index,
        "year": per.year,
        "quarter": per.quarter,
        "quarter_label": [f"{y}-Q{q}" for y, q in zip(per.year, per.quarter)],
        name: s.values
    })
    return out.drop_duplicates(subset=["quarter_end"]).sort_values("quarter_end")

# ---------- CPI (monthly -> YoY monthly -> YoY quarterly) ----------
cpi_raw = pd.read_csv(RAW / "CPI_Monthly_Jan_2013_to_Jun_2025.csv")
c_date = _pick(cpi_raw, ('DATE','Date','Month','Period'))
c_val  = _pick(cpi_raw, ('CPI','Index','Value','CPI_COMBINED_RAW2012_100'))
if not c_date or not c_val:
    raise KeyError("CPI: could not identify date/value columns.")

cpi = _parse_dates_in_df(cpi_raw, c_date)
cpi_index_m = cpi.set_index(c_date)[c_val].astype(float).asfreq('M', method='pad')
cpi_yoy_m   = _yoy_from_monthly(cpi_index_m).rename('cpi_yoy_m')
cpi_yoy_q   = _monthly_to_quarter(cpi_yoy_m, 'mean').rename('cpi_yoy')
cpi_df = _standardize_q_series(cpi_yoy_q, "cpi_yoy")
cpi_df.to_csv(PROC / "quarterly_cpi_yoy.csv", index=False)

# ---------- GDP (quarterly levels -> YoY), fiscal-aware (Q-MAR) then normalized to calendar quarter-end stamps ----------
gdp_raw = pd.read_csv(RAW / "GDP_Quarterly_2010_2025.csv").copy()
gdp_raw['gdp_clean'] = pd.to_numeric(gdp_raw['gdp'].astype(str).str.replace(',', '', regex=False), errors='coerce')
qstr_raw = gdp_raw['quarter'].astype(str).str.replace('.0','',regex=False)
m = qstr_raw.str.extract(r'(?P<y>\d{4}).*?Q(?P<q>[1-4])')
qstr = (m['y'].fillna('') + 'Q' + m['q'].fillna(''))
valid = qstr.str.match(r'^\d{4}Q[1-4]$')
q_end_fy = pd.PeriodIndex(qstr[valid], freq='Q-MAR').to_timestamp(how='end')
gdp_level_fy = (pd.Series(gdp_raw.loc[valid,'gdp_clean'].values, index=q_end_fy)
                  .dropna().sort_index().groupby(pd.Grouper(freq='Q')).last())
gdp_yoy = gdp_level_fy.pct_change(4).mul(100.0).rename('gdp_yoy')
gdp_df = _standardize_q_series(gdp_yoy, "gdp_yoy")
gdp_df.to_csv(PROC / "quarterly_gdp_yoy.csv", index=False)

# ---------- Repo (monthly -> quarter level %, and Δ bps) ----------
repo_raw = pd.read_csv(RAW / "Repo_Rate_Monthly_2010_2025.csv")
r_date = _pick(repo_raw, ['DATE','Date','Month','Period'])
r_val  = _pick(repo_raw, ['REPO_RATE_PERCENT','Repo','Rate','Policy Rate','REPO','Value'])
repo = _parse_dates_in_df(repo_raw, r_date)
val = (repo[r_val].astype(str).str.replace('\u200b','',regex=False)
                     .str.replace(',','',regex=False).str.replace('%','',regex=False).str.strip())
val = pd.to_numeric(val, errors='coerce')
if val.dropna().abs().median() < 1:
    val = val * 100.0
repo = repo.assign(val=val).dropna(subset=['val'])
repo_m = (repo.set_index(r_date)['val'].to_period('M').to_timestamp('M').sort_index())
repo_full_m = pd.date_range(repo_m.index.min(), repo_m.index.max(), freq='M')
repo_m = repo_m.reindex(repo_full_m).ffill()
repo_q = repo_m.resample('Q').last().rename('repo')
repo_chg_bps = (repo_q.diff() * 100.0).rename('repo_chg_bps')
repo_df = _standardize_q_series(repo_q, "repo")
repo_chg_df = _standardize_q_series(repo_chg_bps, "repo_chg_bps")
repo_df.to_csv(PROC / "quarterly_repo_level.csv", index=False)
repo_chg_df.to_csv(PROC / "quarterly_repo_chg_bps.csv", index=False)

# ---------- NIFTY & MIDCAP (daily -> quarter returns) ----------
n50 = pd.read_csv(RAW / "Nifty50.csv")
mid = pd.read_csv(RAW / "NIFTYMidcap100.csv")
d_n50 = _pick(n50, ['Date','DATE','date']); p_n50 = _pick(n50, ['Price','Close','Adj Close','AdjClose','Last','PX_LAST'])
d_mid = _pick(mid, ['Date','DATE','date']); p_mid = _pick(mid, ['Price','Close','Adj Close','AdjClose','Last','PX_LAST'])

def _parse_prices(df, dcol, vcol):
    out = _parse_dates_in_df(df, dcol)
    out[vcol] = pd.to_numeric(out[vcol].astype(str).str.replace(',','',regex=False), errors='coerce')
    return out.dropna(subset=[vcol])[[dcol, vcol]]

n50p = _parse_prices(n50, d_n50, p_n50)
midp = _parse_prices(mid, d_mid, p_mid)

n50_q = n50p.set_index(d_n50)[p_n50].resample('Q').last()
mid_q = midp.set_index(d_mid)[p_mid].resample('Q').last()

nifty_qret  = n50_q.pct_change().rename('nifty_qret')
midcap_qret = mid_q.pct_change().rename('midcap_qret')
excess_ret  = (midcap_qret - nifty_qret).rename('excess_ret')

nifty_df  = _standardize_q_series(nifty_qret,  "nifty_qret")
midcap_df = _standardize_q_series(midcap_qret, "midcap_qret")
excess_df = _standardize_q_series(excess_ret,  "excess_ret")

nifty_df.to_csv(PROC / "quarterly_nifty_qret.csv", index=False)
midcap_df.to_csv(PROC / "quarterly_midcap_qret.csv", index=False)
excess_df.to_csv(PROC / "quarterly_excess_ret.csv", index=False)

# ---------- Merged views (outer & inner) ----------
key_cols = ["quarter_end","year","quarter","quarter_label"]
dfs = [excess_df, cpi_df, gdp_df, repo_df, repo_chg_df]

# Outer (union) merge on quarter_end, keeping all keys
merged_outer = dfs[0]
for d in dfs[1:]:
    merged_outer = pd.merge(merged_outer, d, on=key_cols, how="outer")

merged_outer = merged_outer.sort_values("quarter_end").reset_index(drop=True)
merged_outer.to_csv(PROC / "quarterly_merged_outer.csv", index=False)

# Inner (strict intersection)
merged_inner = dfs[0]
for d in dfs[1:]:
    merged_inner = pd.merge(merged_inner, d, on=key_cols, how="inner")
merged_inner = merged_inner.sort_values("quarter_end").reset_index(drop=True)
merged_inner.to_csv(PROC / "quarterly_merged_intersection.csv", index=False)

# Simple coverage summary for each series
def _coverage(df, col):
    nonnull = df[col].notna()
    if nonnull.any():
        idx = df.loc[nonnull, "quarter_label"]
        return idx.iloc[0], idx.iloc[-1], int(nonnull.sum())
    return None, None, 0

coverage = {
    "excess_ret": _coverage(merged_outer, "excess_ret"),
    "cpi_yoy":    _coverage(merged_outer, "cpi_yoy"),
    "gdp_yoy":    _coverage(merged_outer, "gdp_yoy"),
    "repo":       _coverage(merged_outer, "repo"),
    "repo_chg_bps": _coverage(merged_outer, "repo_chg_bps"),
}

coverage


{'excess_ret': ('2010-Q2', '2025-Q3', 62),
 'cpi_yoy': ('2014-Q1', '2025-Q2', 46),
 'gdp_yoy': ('2012-Q2', '2024-Q3', 50),
 'repo': ('2010-Q1', '2025-Q3', 63),
 'repo_chg_bps': ('2010-Q2', '2025-Q3', 62)}

In [57]:
# Validate and finalize the rainfall processing on the uploaded CSV.
# Load
rain = pd.read_csv("raw/AnnualRainfall_with_Good_and_Anomaly_2012_2025.csv")

# Columns expected
ycol, mcol = 'year', 'month'
obs_col, norm_col, mm_anom_col = 'rainfall_mm', 'good_rainfall_mm', 'anomaly_mm'

missing_cols = [c for c in [ycol, mcol, obs_col, norm_col] if c not in rain.columns]
assert not missing_cols, f"Missing columns in rainfall CSV: {missing_cols}. Have: {list(rain.columns)}"

# Month text -> month number
mmap = {m[:3].lower(): i for i, m in enumerate(
    ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], start=1)}

rain['mon'] = rain[mcol].astype(str).str[:3].str.lower().map(mmap)

# QA: detect any unknown month strings
if rain['mon'].isna().any():
    unk = rain.loc[rain['mon'].isna(), mcol].unique().tolist()
    raise ValueError(f"Unrecognized month strings: {unk}")

rain['mon'] = rain['mon'].astype(int)

# Keep Southwest monsoon months: Jun–Sep
mons = rain[rain['mon'].between(6, 9)].copy()

# QA1: ensure each season has 4 months
miss = (mons.groupby(ycol)['mon'].nunique().rename('monsoon_months').reset_index())
bad = miss[miss['monsoon_months'] < 4]
if not bad.empty:
    print("⚠ Monsoon months missing in years:", bad[ycol].tolist())

# Aggregate seasonal totals
grp = mons.groupby(ycol, as_index=False).agg(
    obs_mm  = (obs_col,  'sum'),
    norm_mm = (norm_col, 'sum'),
    anom_sum_mm = (mm_anom_col, 'sum')
)

# Guard against division by zero in norm
grp.loc[grp['norm_mm'] == 0, 'norm_mm'] = np.nan

# IMD-style anomaly %
grp['rain_anom_pct'] = (grp['obs_mm'] - grp['norm_mm']) / grp['norm_mm'] * 100.0

# Cross-check sign using anomaly_mm if provided as (good - observed)
grp['alt_pct_from_file'] = (-grp['anom_sum_mm'] / grp['norm_mm']) * 100.0

# Stamp each year's anomaly at Sep-30; forward-fill within quarters
rain_idx = pd.to_datetime(grp[ycol].astype(int).astype(str) + '-09-30')
rain_q = pd.Series(grp['rain_anom_pct'].values, index=rain_idx).resample('Q').ffill().rename('rain_anom')

# Standardize to labeled DataFrame for joining
per = rain_q.index.to_period('Q')
rain_df = pd.DataFrame({
    "quarter_end": rain_q.index.to_period('Q').to_timestamp('Q'),
    "year": per.year,
    "quarter": per.quarter,
    "quarter_label": [f"{y}-Q{q}" for y, q in zip(per.year, per.quarter)],
    "rain_anom": rain_q.values
}).drop_duplicates(subset=["quarter_end"]).sort_values("quarter_end")

# Save outputs
out_season = grp.copy()
out_season.to_csv("processed/rainfall_seasonal_totals.csv", index=False)
rain_df.to_csv("processed/quarterly_rain_anom.csv", index=False)

# Display quick previews
print("Rainfall seasonal totals (QA)", out_season.head(6))
print("Rainfall quarterly anomaly (head)", rain_df.head(12))
print("Rainfall quarterly anomaly (tail)", rain_df.tail(12))

print("Rainfall seasonal years covered:", grp[ycol].tolist())
print("Rainfall quarterly points:", rain_q.shape[0], "| Range:", rain_q.index.min().date(), "→", rain_q.index.max().date())


Rainfall seasonal totals (QA)    year  obs_mm  norm_mm  anom_sum_mm  rain_anom_pct  alt_pct_from_file
0  2012  1061.5    868.5       -193.0      22.222222          22.222222
1  2013  1475.4    868.5       -606.9      69.879102          69.879102
2  2014   803.8    868.5         64.7      -7.449626          -7.449626
3  2015   814.9    868.5         53.6      -6.171560          -6.171560
4  2016  1347.9    868.5       -479.4      55.198618          55.198618
5  2017   845.9    868.5         22.6      -2.602188          -2.602188
Rainfall quarterly anomaly (head)    quarter_end  year  quarter quarter_label  rain_anom
0   2012-09-30  2012        3       2012-Q3  22.222222
1   2012-12-31  2012        4       2012-Q4  22.222222
2   2013-03-31  2013        1       2013-Q1  22.222222
3   2013-06-30  2013        2       2013-Q2  22.222222
4   2013-09-30  2013        3       2013-Q3  69.879102
5   2013-12-31  2013        4       2013-Q4  69.879102
6   2014-03-31  2014        1       2014-Q1  69

In [58]:
files = [
  "quarterly_excess_ret.csv","quarterly_cpi_yoy.csv","quarterly_gdp_yoy.csv",
  "quarterly_repo_level.csv","quarterly_repo_chg_bps.csv","quarterly_rain_anom.csv"
]

for fn in files:
    df = pd.read_csv(PROC / fn, parse_dates=['quarter_end'])

    # derive year/quarter from the quarter_end column
    per = df['quarter_end'].dt.to_period('Q')          # Series[period[Q]]
    y = per.dt.year.to_numpy()                          # <-- use .dt.year
    q = per.dt.quarter.to_numpy()                       # <-- use .dt.quarter

    # make sure the stored cols are ints (in case they were read as object)
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df['quarter'] = pd.to_numeric(df['quarter'], errors='coerce').astype('Int64')

    # checks
    assert (df['year'].to_numpy() == y).all(), f"{fn}: year mismatch"
    assert (df['quarter'].to_numpy() == q).all(), f"{fn}: quarter mismatch"

    expected_labels = pd.Series([f"{yy}-Q{qq}" for yy, qq in zip(y, q)], index=df.index)
    assert (df['quarter_label'] == expected_labels).all(), f"{fn}: quarter_label mismatch"

print("All quarter/year labels are consistent.")

All quarter/year labels are consistent.


In [ ]:
BASE = Path("./processed")

# ---- helper to load a standardized quarterly CSV and return a Series ----
def load_std_series(path: Path) -> pd.Series:
    df = pd.read_csv(path, parse_dates=['quarter_end'])
    key_cols = ["quarter_end","year","quarter","quarter_label"]
    val_cols = [c for c in df.columns if c not in key_cols]
    if len(val_cols) != 1:
        raise ValueError(f"{path.name}: expected exactly 1 value column, found {val_cols}")
    name = val_cols[0]
    s = pd.Series(df[name].values, index=df['quarter_end']).rename(name)
    # normalize to calendar quarter-end timestamps
    s.index = s.index.to_period('Q').to_timestamp('Q')
    return s

# ---- load available series ----
series_paths = {
    "excess_ret": BASE / "quarterly_excess_ret.csv",
    "cpi_yoy": BASE / "quarterly_cpi_yoy.csv",
    "gdp_yoy": BASE / "quarterly_gdp_yoy.csv",
    "repo": BASE / "quarterly_repo_level.csv",
    "repo_chg_bps": BASE / "quarterly_repo_chg_bps.csv",
    "rain_anom": BASE / "quarterly_rain_anom.csv",  
}

series = {}
missing = []
for k, p in series_paths.items():
    try:
        series[k] = load_std_series(p)
    except FileNotFoundError:
        missing.append(k)
    except Exception as e:
        print(f"⚠ Issue loading {k} from {p.name}: {e}")
        missing.append(k)

if missing:
    print("Note: Missing series not used in modeling:", missing)

# ---- merge into a single DataFrame (inner join on intersection) ----
df_all = pd.concat(series.values(), axis=1, join='inner').sort_index()
df_all.index.name = "quarter_end"
# carry forward rainfall across Q3→next Q2 already done during build; we just keep as-is
# Quick summary
print("Merged intersection rows:", df_all.shape[0], "| range:", df_all.index.min().date(), "→", df_all.index.max().date())
print("Columns:", list(df_all.columns))

# Save the base merged file for reference
df_all_out = df_all.copy()
df_all_out["year"] = df_all_out.index.to_period('Q').year
df_all_out["quarter"] = df_all_out.index.to_period('Q').quarter
df_all_out["quarter_label"] = [f"{y}-Q{q}" for y, q in zip(df_all_out["year"], df_all_out["quarter"])]
df_all_out.reset_index().to_csv(BASE / "quarterly_merged_for_modeling.csv", index=False)

# ---- Option A: Nowcast next quarter with current-quarter info ----
dfa = df_all.copy()
dfa["ret_prev_q"]      = dfa["excess_ret"].shift(1)       # baseline feature
# keep contemporaneous macro info (current quarter)
# (if some series are missing from 'series', skip them gracefully)
feat_cols_A = ["ret_prev_q"]
for col in ["cpi_yoy","gdp_yoy","repo_chg_bps","rain_anom"]:
    if col in dfa.columns:
        # contemporaneous variables for quarter t
        dfa[col + "_cur"] = dfa[col]
        feat_cols_A.append(col + "_cur")

dfa["excess_next_q"] = dfa["excess_ret"].shift(-1)        # target at t+1

dfa_model = dfa[feat_cols_A + ["excess_next_q"]].dropna().copy()
dfa_model.index.name = "quarter_end"
dfa_model.reset_index().to_csv(BASE / "modeling_nowcast_optionA.csv", index=False)

print("\nOption A (nowcast) rows:", dfa_model.shape[0], "| features:", feat_cols_A)

# ---- Add small, readable previews for both designs ----
print("Modeling dataset (Option A, head)", dfa_model.reset_index().head(10))

# ---- Create CV fold summaries (expanding window) for each design ----
def make_expanding_cv(index, min_train=24, test_size=2, step=2):
    """
    index: DatetimeIndex (quarter_end) sorted.
    Returns a DataFrame with fold_id, train_start, train_end, test_start, test_end.
    """
    idx = pd.Index(index)  # ensure index
    n = len(idx)
    folds = []
    train_end = min_train
    fold_id = 1
    while train_end < n:
        test_end = min(train_end + test_size, n)
        if test_end - train_end <= 0:
            break
        folds.append({
            "fold_id": fold_id,
            "train_start": idx[0].date(),
            "train_end": idx[train_end-1].date(),
            "test_start": idx[train_end].date(),
            "test_end": idx[test_end-1].date()
        })
        fold_id += 1
        train_end += step
    return pd.DataFrame(folds)
folds_A = make_expanding_cv(dfa_model.index, min_train=24, test_size=2, step=2)
folds_A.to_csv(BASE / "cv_folds_nowcast_optionA.csv", index=False)
print("CV folds (Option A summary)", folds_A.head(10))


print("\nSaved:")
print("quarterly_merged_for_modeling_optionA.csv")
print("modeling_nowcast_optionA.csv")
print("cv_folds_nowcast_optionA.csv")

Merged intersection rows: 43 | range: 2014-03-31 → 2024-09-30
Columns: ['excess_ret', 'cpi_yoy', 'gdp_yoy', 'repo', 'repo_chg_bps', 'rain_anom']

Option A (nowcast) rows: 41 | features: ['ret_prev_q', 'cpi_yoy_cur', 'gdp_yoy_cur', 'repo_chg_bps_cur', 'rain_anom_cur']
Modeling dataset (Option A, head)   quarter_end  ret_prev_q  cpi_yoy_cur  gdp_yoy_cur  repo_chg_bps_cur  \
0  2014-06-30    0.003563     7.859486     7.112080               0.0   
1  2014-09-30    0.153161     6.681568     7.592544               0.0   
2  2014-12-31   -0.017474     4.054538     8.033806               0.0   
3  2015-03-31    0.062164     5.272440     7.197433             -50.0   
4  2015-06-30    0.008021     5.090809     9.088683             -25.0   
5  2015-09-30    0.015073     3.948304     8.679803             -50.0   
6  2015-12-31    0.048207     5.339795     9.669809               0.0   
7  2016-03-31    0.032066     5.259609     8.575494               0.0   
8  2016-06-30   -0.021910     5.665680   

In [63]:
# ---- Option B: Forecast current quarter with last-quarter info ----
dfb = df_all.copy()
dfb["ret_prev_q"]       = dfb["excess_ret"].shift(1)      # baseline
feat_cols_B = ["ret_prev_q"]
for col in ["cpi_yoy","gdp_yoy","repo_chg_bps","rain_anom"]:
    if col in dfb.columns:
        dfb[col + "_lag"] = dfb[col].shift(1)             # use last quarter info
        feat_cols_B.append(col + "_lag")

dfb["excess_curr_q"] = dfb["excess_ret"]                  # target at t

dfb_model = dfb[feat_cols_B + ["excess_curr_q"]].dropna().copy()
dfb_model.index.name = "quarter_end"
dfb_model.reset_index().to_csv(BASE / "modeling_forecast_optionB.csv", index=False)

print("Option B (forecast) rows:", dfb_model.shape[0], "| features:", feat_cols_B)

# ---- Add small, readable previews for both designs ----
print("Modeling dataset (Option A, head)", dfa_model.reset_index().head(10))
print("Modeling dataset (Option B, head)", dfb_model.reset_index().head(10))

# ---- Create CV fold summaries (expanding window) for each design ----
def make_expanding_cv(index, min_train=24, test_size=2, step=2):
    """
    index: DatetimeIndex (quarter_end) sorted.
    Returns a DataFrame with fold_id, train_start, train_end, test_start, test_end.
    """
    idx = pd.Index(index)  # ensure index
    n = len(idx)
    folds = []
    train_end = min_train
    fold_id = 1
    while train_end < n:
        test_end = min(train_end + test_size, n)
        if test_end - train_end <= 0:
            break
        folds.append({
            "fold_id": fold_id,
            "train_start": idx[0].date(),
            "train_end": idx[train_end-1].date(),
            "test_start": idx[train_end].date(),
            "test_end": idx[test_end-1].date()
        })
        fold_id += 1
        train_end += step
    return pd.DataFrame(folds)

folds_B = make_expanding_cv(dfb_model.index, min_train=24, test_size=2, step=2)
folds_B.to_csv(BASE / "cv_folds_forecast_optionB.csv", index=False)
print("CV folds (Option B summary)", folds_B.head(10))

print("\nSaved:")
print("quarterly_merged_for_modeling_optionB.csv")
print("modeling_forecast_optionB.csv")
print("cv_folds_forecast_optionB.csv")

Option B (forecast) rows: 42 | features: ['ret_prev_q', 'cpi_yoy_lag', 'gdp_yoy_lag', 'repo_chg_bps_lag', 'rain_anom_lag']
Modeling dataset (Option A, head)   quarter_end  ret_prev_q  cpi_yoy_cur  gdp_yoy_cur  repo_chg_bps_cur  \
0  2014-06-30    0.003563     7.859486     7.112080               0.0   
1  2014-09-30    0.153161     6.681568     7.592544               0.0   
2  2014-12-31   -0.017474     4.054538     8.033806               0.0   
3  2015-03-31    0.062164     5.272440     7.197433             -50.0   
4  2015-06-30    0.008021     5.090809     9.088683             -25.0   
5  2015-09-30    0.015073     3.948304     8.679803             -50.0   
6  2015-12-31    0.048207     5.339795     9.669809               0.0   
7  2016-03-31    0.032066     5.259609     8.575494               0.0   
8  2016-06-30   -0.021910     5.665680     6.289328             -25.0   
9  2016-09-30    0.012432     5.167334     6.112753               0.0   

   rain_anom_cur  excess_next_q  
0    